In [1]:
import os, glob, re, string
import numpy as np
import pandas as pd

### Import the Complete Lord of the Rings Text

In [1]:
# complete texts of Lord of the Rings can be located at 
# 'https://archive.org/details/TheLordOfTheRing1TheFellowshipOfTheRing'

In [2]:
with open('./the_lord_of_the_rings_text/Lord_of_the_Rings_complete.txt', 'r') as file:
    lotr = file.read().replace('\n', '')

In [3]:
# check the size of the string to double check
len(lotr)

2512368

In [4]:
# Leaving the corpus as one long string is optimal for NLP, LDA, and NMF

### Prepare Text for NLP (Tokenize, Punctuation Removal, Stopwords) 

In [6]:
import nltk;
from nltk.corpus import stopwords;
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import RegexpTokenizer, sent_tokenize


import sklearn;
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer;

# Please see my 'preprocessing_stopwords_plotly.ipynb' notebook for complete annotated preprocessing steps

In [7]:
# remove punctuation and tokenize

tokenizer = RegexpTokenizer(r'\w+')
lotr_tokens = tokenizer.tokenize(lotr)

# Keeping capitalization because I want the model to treat the proper nouns accordingly. Names are important in LotR. 

In [8]:
lotr_tokens[2019]

'Hobbits'

In [9]:
# Based on previous models, it's imperative to add some stop words. 
# add stopwords - changing this list can have a dramatic effect on results in the LDA model because it uses word counts

stopwords = nltk.corpus.stopwords.words('english')
stopwords.append('said') # removed due to disproportional frequency
stopwords.append('come')
stopwords.append('came')

# Additional stopwords, like proper names, could dramatically alter results. Keeping them to preserve the text. 

In [10]:
# Removes stopwords 
lotr_clean = [word for word in lotr_tokens if word.lower() not in stopwords]
print("="*90)
print(f'Length of original list: {len(lotr_tokens)} words\n')
print(f'Length of list after stopwords removal: {len(lotr_clean)} words')

Length of original list: 482056 words

Length of list after stopwords removal: 221549 words


In [11]:
lotr_clean[1891]

'flowers'

In [14]:
# Lemmatize tokens.
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()
lotr_tokens_lems = [lemmatizer.lemmatize(i) for i in lotr_clean]

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/jdhazard/nltk_data...


In [15]:
lotr_tokens_lems[1891]

'flower'

### Combined Lemmmatizing and CountVectorizer into one Class

In [16]:
# Graciously borrowed from 'https://www.kaggle.com/meiyizi/spooky-nlp-and-topic-modelling-tutorial' 
# We have essentially inherited and subclassed the original Sklearn's CountVectorizer class 
# and overwritten the build_analyzer method by implementing the lemmatizer for each list in the raw text matrix.

lemm = WordNetLemmatizer()
class LemmaCountVectorizer(CountVectorizer):
    def build_analyzer(self):
        analyzer = super(LemmaCountVectorizer, self).build_analyzer()
        return lambda doc: (lemm.lemmatize(w) for w in analyzer(doc))

In [17]:
# Calling our overwritten Count vectorizer
tf_vectorizer = LemmaCountVectorizer(max_df=0.95, 
                                     min_df=2,
                                     stop_words='english',
                                     decode_error='ignore')
tf = tf_vectorizer.fit_transform(lotr_tokens_lems)

### Build Our LDA Model

In [18]:
import time

from sklearn.decomposition import NMF;
from sklearn.preprocessing import normalize;
from sklearn.decomposition import LatentDirichletAllocation

# Using the SKlearn LDA and NMF models -- The Gensim topic models produced uninterpreable results

In [19]:
lda = LatentDirichletAllocation(n_components=21, max_iter=5,
                                learning_method = 'online',
                                learning_offset = 50.,
                                random_state = 2019)

In [20]:
lda.fit(tf)
print(time.time())

1779211772.0088


In [21]:
# Graciously borrowed from 'https://www.kaggle.com/meiyizi/spooky-nlp-and-topic-modelling-tutorial'
# Define helper function to print top words

def print_top_words(model, feature_names, n_top_words):
    for index, topic in enumerate(model.components_): # enumerate keeps count of iterations 
        message = "\nTopic #{}:".format(index) 
        message += " ".join([feature_names[i] for i in topic.argsort()[:-n_top_words - 1 :-1]])
        # list comprehension inside .join function
        print(message)
        print("="*70) # prints 70 '=' signs as separators

In [23]:
n_top_words = 20
print("\nTopics in LDA model: ")
tf_feature_names = tf_vectorizer.get_feature_names_out()
print_top_words(lda, tf_feature_names, n_top_words)


Topics in LDA model: 

Topic #0:hobbit look place gimli sun rode felt boromir fall mina mile open sauron treebeard ents hole sure met prince eastern

Topic #1:way foot set gone enemy brought wood longer horn strange plain window country smoke edge bent mouth creature nice weapon

Topic #2:long gandalf hand light mr man west darkness elf answer walked forth short bring breath noise past deadly failed leader

Topic #3:men door east sleep lady cold strength food sight doom hard grass slope order try cliff tidings lead arrow servant

Topic #4:great turned house folk dead coming fair knew wonder gave sky silver chief remember farewell bag father bank falling room

Topic #5:little hope soon yes took saruman rohan battle red called star tirith eastward wait memory dwarf sign unless worse gloom

Topic #6:wall north near evil world clear guard tale death step work shape feel running poor young dream companion matter length

Topic #7:went dark lord thought tower company path river rest leaf cou

### Build Our NMF Model

#### Non-Negative Matrix Factorization comparison to LDA

In [24]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.decomposition import NMF
from sklearn.preprocessing import normalize

In [25]:
# Use CountVectorizer to get total word counts in documents

vectorizer = CountVectorizer(analyzer = "word", max_features = 10_000)

In [26]:
# Transform words with TfidfTransformer - This takes into account term frequency across and within documents

word_counts = vectorizer.fit_transform(lotr_tokens_lems)

tfidf_transform = TfidfTransformer(smooth_idf = False)

words_tfidf = tfidf_transform.fit_transform(word_counts)

# final_words = normalize(words_tfidf, norm = 'l1')

In [27]:
# Instantiate NMF model and fit to tfidf transformed documents

model = NMF(n_components = 50, init = 'nndsvd')

# Set W as the document by topic matrix
# Set H as the topic by word matrix

W = model.fit_transform(words_tfidf)
H = model.components_

In [28]:
# Function to assign topic indices back to feature names - takes model, feature names from vectorizer, 
# and n_top_words as arguments. n_top_words selects the number of keywords per topic
# renders differently than our previous function essentially does the same thing 
# Thanks to Nick Gayliard for inspiration 

def print_top_words(model, feature_names, n_top_words):
    lst = []
    for topic_idx, topic in enumerate(model.components_):
        message = "Topic #%d: " % topic_idx
        message += " ".join([feature_names[i]
                             for i in topic.argsort()[:-n_top_words - 1:-1]])
        lst.append(message)
    return lst

In [30]:
# List numbered topic and top 10 words in the topic 

topics_nmf = (print_top_words(model, vectorizer.get_feature_names_out(), 10))

In [31]:
# Check nmf topic assignment

topics_nmf

['Topic #0: frodo saw stood end shadow think ever black heard foot',
 'Topic #1: sam saw end think foot heard left though made king',
 'Topic #2: great behind ever heard stone though left king first going',
 'Topic #3: could end ever heard foot stone voice side hill let',
 'Topic #4: long stood foot side passed voice made hill gollum fell',
 'Topic #5: would end think saw ever foot heard left let lord',
 'Topic #6: like saw shadow stood black though passed first hill white',
 'Topic #7: gandalf good stone though side left made king water hill',
 'Topic #8: go behind good ever foot stone side voice passed left',
 'Topic #9: one good end foot stone side much two gollum found',
 'Topic #10: back saw good think king let water hill lord master',
 'Topic #11: away saw end stood think black heard foot though passed',
 'Topic #12: still saw shadow stood end heard black side though made',
 'Topic #13: see end good stone passed side foot left voice white',
 'Topic #14: many saw shadow think voic

### Conclusions: 
### Topic Modelling for works of fiction requires further research. My hypothesis is that given the assumptions made by both the LDA and NMF models easily interpretable results are both difficult to reach and hard to interpret. Close examination of these assumptions, word counts following a specific distribution, for example,  should be made before continuing with this line of questioning. 